In [1]:
from pymongo import MongoClient
import polars as pl
import pandas as pd

In [4]:
df_lyon = pl.read_csv("listings_Lyon.csv")
df_paris= pl.read_csv("listings_Paris.csv")

In [5]:
df_lyon = df_lyon.with_columns(pl.lit("Lyon").alias("ville"))
df_paris = df_paris.with_columns(pl.lit("Paris").alias("ville"))

In [6]:
df_paris.write_csv("paris_with_ville.csv")
df_lyon.write_csv("lyon_with_ville.csv")

In [7]:
dfl=pd.read_csv('lyon_with_ville.csv')
dfp=pd.read_csv('paris_with_ville.csv')

In [8]:
def nettoyer_data(df):
    colonnes_a_supprimer = [
        "description", 
        "neighborhood_overview", 
        "host_about",
        "picture_url",
        "host_thumbnail_url", 
        "host_picture_url",
        "listing_url",
        "host_url"
    ]
    df_clean = df.drop(columns=colonnes_a_supprimer, errors="ignore")

    for col in df_clean.columns:
        if df_clean[col].dtype == "object":
            df_clean[col] = (df_clean[col].astype(str)
                .str.replace('"', '', regex=False)
                .str.replace('\n', ' ', regex=False)
                .str.replace('\r', ' ', regex=False)
                .str.replace(',', ' ', regex=False))
    
    if "price" in df_clean.columns:
        df_clean["price"] = (df_clean["price"]
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.replace(" ", "", regex=False)
            .astype(float))
    
    return df_clean

In [9]:
df_paris_clea = nettoyer_data(dfp)
df_lyon_clean = nettoyer_data(dfl)

In [10]:
df_paris_clea.to_csv("paris_clean.csv", index=False, encoding="utf-8-sig")
df_lyon_clean.to_csv("lyon_clean.csv", index=False, encoding="utf-8-sig")